In [30]:
import numpy as np
from matplotlib import pyplot as plt
import pickle
import wandb

In [17]:
def mean_geometric(array):

	log_array = np.log(array)
	log_mean = np.mean(log_array)
	geo_mean = np.exp(log_mean)

	return geo_mean

In [22]:
api = wandb.Api()

username = 'julian-zim-uni-vie'

projects = {
	'll00' : {
		'hgm' : ['hgm-ll0-02-12-01h36-0-2022-11-23_worm10',
				 'hgm-ll0-02-12-16h16-1-2022-11-23_worm11',
				 'hgm-ll0-03-12-06h26-7-2023-11-21_14-46_wt_worm4-2023-11-21',
				 'hgm-ll0-03-12-20h22-11-2023-12-19_17-35_wt_FM_worm4-2023-12-19',
				 'hgm-ll0-04-12-11h04-12-2023-12-19_18-27_wt_FM_worm5again2-2023-12-19',
				 'hgm-ll0-05-12-08h57-17-ZIM2165_Gcamp7b_worm1-2022-11-30',
				 'hgm-ll0-01-12-11h07-19-ZIM2165_Gcamp7b_worm1-2022_11_28',
				 'hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30',],
		'bom' : ['bom-ll0 02.12 08h22 0 2022-11-23_worm10',
				 'bom-ll0 03.12 06h24 1 2022-11-23_worm11',
				 'bom-ll0 04.12 03h09 7 2023-11-21_14-46_wt_worm4-2023-11-21',
				 'bom-ll0 05.12 01h10 11 2023-12-19_17-35_wt_FM_worm4-2023-12-19',
				 'bom-ll0 05.12 23h09 12 2023-12-19_18-27_wt_FM_worm5again2-2023-12-19',
				 'bom-ll0 06.12 21h03 17 ZIM2165_Gcamp7b_worm1-2022-11-30',
				 'bom-ll0 01.12 11h07 19 ZIM2165_Gcamp7b_worm1-2022_11_28',
				 'bom-ll0 07.12 20h51 21 ZIM2165_Gcamp7b_worm2-2022-11-30',],
	},
	'll67' : {
		'hgm' : ['hgm-ll67 02.12 09h08 0 2022-11-23_worm10',
				 'hgm-ll67 03.12 09h17 1 2022-11-23_worm11',
				 'hgm-ll67 04.12 04h25 7 2023-11-21_14-46_wt_worm4-2023-11-21',
				 'hgm-ll67 05.12 00h24 11 2023-12-19_17-35_wt_FM_worm4-2023-12-19',
				 'hgm-ll67 05.12 16h37 12 2023-12-19_18-27_wt_FM_worm5again2-2023-12-19',
				 'hgm-ll67 06.12 12h51 17 ZIM2165_Gcamp7b_worm1-2022-11-30',
				 'hgm-ll67 01.12 11h08 19 ZIM2165_Gcamp7b_worm1-2022_11_28',
				 'hgm-ll67 07.12 12h03 21 ZIM2165_Gcamp7b_worm2-2022-11-30',],
		'bom' : ['bom-ll67 01.12 23h25 0 2022-11-23_worm10',
				 'bom-ll67 02.12 12h58 1 2022-11-23_worm11',
				 'bom-ll67 02.12 23h57 7 2023-11-21_14-46_wt_worm4-2023-11-21',
				 'bom-ll67 03.12 12h25 11 2023-12-19_17-35_wt_FM_worm4-2023-12-19',
				 'bom-ll67 03.12 23h53 12 2023-12-19_18-27_wt_FM_worm5again2-2023-12-19',
				 'bom-ll67 04.12 12h34 17 ZIM2165_Gcamp7b_worm1-2022-11-30',
				 'bom-ll67 01.12 11h07 19 ZIM2165_Gcamp7b_worm1-2022_11_28',
				 'bom-ll67 05.12 02h35 21 ZIM2165_Gcamp7b_worm2-2022-11-30',],
	},
}

hyperparams = {
	'll00' : {
		'hgm' : {
			'w1_reg_lambda_mean': {
				'linear': -1.,
				'geometric': -1.,
			},
			'w2_reg_lambda_mean': {
				'linear': -1.,
				'geometric': -1.,
			},
		},
		'bom' : {
			'w1_reg_lambda_mean': {
				'linear': -1.,
				'geometric': -1.,
			},
		},
	},
	'll67' : {
		'hgm' : {
			'w1_reg_lambda_mean': {
				'linear': -1.,
				'geometric': -1.,
			},
			'w2_reg_lambda_mean': {
				'linear': -1.,
				'geometric': -1.,
			},
		},
		'bom' : {
			'w1_reg_lambda_mean': {
				'linear': -1.,
				'geometric': -1.,
			},
		},
	},
}

In [23]:
for ll in ['ll00', 'll67']:
	for model in ['bom', 'hgm']:
		for project in projects[ll][model]:
			if len(project) == 0:
				continue

			print(f'getting runs for {project}...')
			runs = api.runs(f'{username}/{project}')

			print('sorting them...')
			sorted_runs = sorted(runs, key=lambda run: run.summary.get('mpe_loss', float('inf')))
			best_runs = sorted_runs[:int(0.05 * len(sorted_runs))]

			print(f'{model} w1 averaging...')
			best_w1_reg_lambdas = np.array([run.config['w1_reg_lambda'] for run in best_runs])
			w1_reg_lambdas_mean_lin = np.mean(best_w1_reg_lambdas)
			w1_reg_lambdas_mean_geo = mean_geometric(best_w1_reg_lambdas)
			hyperparams[ll][model]['w1_reg_lambda_mean']['linear'] = w1_reg_lambdas_mean_lin
			hyperparams[ll][model]['w1_reg_lambda_mean']['geometric'] = w1_reg_lambdas_mean_geo

			if model == 'hgm':
				print(f'{model} w2 averaging...')
				best_w2_reg_lambdas = np.array([run.config['w2_reg_lambda'] for run in best_runs])
				w2_reg_lambdas_mean_lin = np.mean(best_w2_reg_lambdas)
				w2_reg_lambdas_mean_geo = mean_geometric(best_w1_reg_lambdas)
				hyperparams[ll][model]['w2_reg_lambda_mean']['linear'] = w2_reg_lambdas_mean_lin
				hyperparams[ll][model]['w2_reg_lambda_mean']['geometric'] = w2_reg_lambdas_mean_geo

getting runs for bom-ll0 02.12 08h22 0 2022-11-23_worm10...
sorting them...
bom w1 averaging...
getting runs for bom-ll0 03.12 06h24 1 2022-11-23_worm11...
sorting them...
bom w1 averaging...
getting runs for bom-ll0 04.12 03h09 7 2023-11-21_14-46_wt_worm4-2023-11-21...
sorting them...
bom w1 averaging...
getting runs for bom-ll0 05.12 01h10 11 2023-12-19_17-35_wt_FM_worm4-2023-12-19...
sorting them...
bom w1 averaging...
getting runs for bom-ll0 05.12 23h09 12 2023-12-19_18-27_wt_FM_worm5again2-2023-12-19...
sorting them...
bom w1 averaging...
getting runs for bom-ll0 06.12 21h03 17 ZIM2165_Gcamp7b_worm1-2022-11-30...
sorting them...
bom w1 averaging...
getting runs for bom-ll0 01.12 11h07 19 ZIM2165_Gcamp7b_worm1-2022_11_28...
sorting them...
bom w1 averaging...
getting runs for bom-ll0 07.12 20h51 21 ZIM2165_Gcamp7b_worm2-2022-11-30...
sorting them...
bom w1 averaging...
getting runs for hgm-ll0-02-12-01h36-0-2022-11-23_worm10...
sorting them...
hgm w1 averaging...
hgm w2 averaging.

In [28]:
for model in ['bom', 'hgm']:
	for ll in ['ll00', 'll67']:
		for param in ['w1_reg_lambda_mean', 'w2_reg_lambda_mean']:
			for _type in ['linear', 'geometric']:
				if param in hyperparams[ll][model].keys():
					print(f'{ll} {model} {param} {_type}:\n\t{hyperparams[ll][model][param][_type]:2E}')

ll00 bom w1_reg_lambda_mean linear:
	1.200939E-02
ll00 bom w1_reg_lambda_mean geometric:
	1.196278E-02
ll67 bom w1_reg_lambda_mean linear:
	5.732431E-02
ll67 bom w1_reg_lambda_mean geometric:
	5.724104E-02
ll00 hgm w1_reg_lambda_mean linear:
	1.016857E-03
ll00 hgm w1_reg_lambda_mean geometric:
	9.626962E-04
ll00 hgm w2_reg_lambda_mean linear:
	1.255918E-07
ll00 hgm w2_reg_lambda_mean geometric:
	9.626962E-04
ll67 hgm w1_reg_lambda_mean linear:
	1.025499E-02
ll67 hgm w1_reg_lambda_mean geometric:
	1.010577E-02
ll67 hgm w2_reg_lambda_mean linear:
	3.995751E-07
ll67 hgm w2_reg_lambda_mean geometric:
	1.010577E-02


In [31]:
with open('./hyperparams.pkl', 'wb') as f:
    pickle.dump(hyperparams, f)